<a href="https://colab.research.google.com/github/CleavDcos/Chat-With-Docs-using-Open-Source-HF-LLMs-with-Gradio/blob/main/Chat_With_Docs_using_HF_Open_Source_LLMs_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#iinstall libraries
!pip install -q transformers accelerate bitsandbytes torch pypdf gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.5/334.5 kB 28.7 MB/s eta 0:00:00


In [ ]:
import torch
import gradio as gr
from IPython.display import display, Markdown
import pypdf


In [ ]:
import os
from huggingface_hub import login,notebook_login
print("Attempting to login ton HF...")
notebook_login()
print("Done")

Attempting to login ton HF...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Done


In [ ]:
# Check if GPU is available (essential for running these models)
# Why GPU is Important: LLMs involve billions of calculations (matrix multiplications).
# GPUs are designed for massive parallel processing, making these calculations thousands of times faster than a standard CPU.
# Running these models on a CPU would take an impractically long time (hours for a single answer instead of seconds/minutes).
if torch.cuda.is_available():
    print(f"GPU detected: {torch.cuda.get_device_name(0)}")
    # Set default device to GPU
    torch.set_default_device("cuda")
    print("PyTorch default device set to CUDA (GPU).")
else:
    print("WARNING: No GPU detected. Running these models on CPU will be extremely slow!")
    print("Make sure 'GPU' is selected in Runtime > Change runtime type.")

GPU detected: Tesla T4
PyTorch default device set to CUDA (GPU).


In [ ]:
# Helper function for markdown display
def print_markdown(text):
    """Displays text as Markdown in Colab/Jupyter."""
    display(Markdown(text))

### 📄 Attention Is All You Need  
[Read the paper](https://arxiv.org/abs/1706.03762)

In [ ]:
# The pipelines are a great and easy way to use models for inference.
# These pipelines are objects that abstract most of the complex code from the library, offering a simple API dedicated to several tasks
# Those tasks include Named Entity Recognition, Masked Language Modeling, Sentiment Analysis, Feature Extraction and Question Answering.
from transformers import pipeline

# Load a sentiment classifier model on financial news data
# Check the model here: https://huggingface.co/ProsusAI/finbert
pipe = pipeline(model = "ProsusAI/finbert")
#pipe("Apple lost 10 Million dollars today due to US tarrifs")

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

- **`transformers`:** A Python library that provides a standardized way to download, load, and use models from the Hub with just a few lines of code. Key classes:
    *   `pipeline()`: A high-level, easy-to-use abstraction for common tasks (like text generation, summarization). Great for quick tests and beginners.
    *   `AutoTokenizer`: Automatically downloads the correct "tokenizer" for a model. A tokenizer converts human-readable text into numerical IDs the model understands.
    *   `AutoModelFor...`: Automatically downloads the correct model architecture and pre-trained weights (e.g., `AutoModelForCausalLM` for text generation models like GPT, Llama, Gemma).
- **Other Libraries:** HF also develops libraries like `accelerate` (for efficient loading/distributed training), `datasets` (for handling datasets), and `evaluate` (for model evaluation metrics).



In [ ]:
from transformers import AutoTokenizer

#load tokenizer for gpt 2, this will load the specific tokenizer for gpt 2
tokenizer = AutoTokenizer.from_pretrained("gpt2")
#encode texts to token IDs
tokens = tokenizer("Hello everyone and welcome to LLM and AI Agents Bootcamp")
print(tokens['input_ids'])


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

[15496, 2506, 290, 7062, 284, 27140, 44, 290, 9552, 28295, 18892, 16544]


In [ ]:
!pip install -U bitsandbytes

In [ ]:

!pip install transformers accelerate bitsandbytes

In [ ]:
# Let's import AutoModelForCasualLM

from transformers import AutoModelForCausalLM, BitsAndBytesConfig

# Let's choose a small, powerful model suitable for Colab.
# Alternatives you could try (might need login/agreement):
# model_id = "unsloth/gemma-3-4b-it-GGUF"
# model_id = "Qwen/Qwen2.5-3B-Instruct"
model_id = "Qwen/Qwen2.5-3B-Instruct"
# model_id = "unsloth/Llama-3.2-3B-Instruct"

In [ ]:
# Let's load the Tokenizer
# The tokenizer prepares text input for the model
# trust_remote_code=True is sometimes needed for newer models with custom code.
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    trust_remote_code=True,
    use_fast=False

)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
# Let's Load the Model with Quantization
#We need to quantize the model we r using, for efficient memory usage, can quantize to 8 or 4 bits, rather then the entire model
import torch
print(f"Loading model: {model_id}")
print("This might take a few minutes, especially the first time...")

# Create BitsAndBytesConfig for 4-bit quantization
from transformers import BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

# Load the model with the quantization config
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto"
)

Loading model: Qwen/Qwen2.5-3B-Instruct
This might take a few minutes, especially the first time...


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
prompt = "Explain how electric vehicles work in a funny way"

In [ ]:
#Method 1 to pass to LLM to give output -> Use tokenizing, pass numerical values

inputs = tokenizer(prompt, return_tensors = "pt")

#generate output
outputs = model.generate(**inputs, max_new_tokens = 1000)

response = tokenizer.decode(outputs[0], skip_special_tokens = True)



In [ ]:
#Method 2: to create a pipeline which includes model and tokenizer
#pipeline wraps text generation, tokenization and decoding

pipe = pipeline("text-generation",
                model = model,
                tokenizer = tokenizer,
                torch_dtype = "auto", # Match model dtype
                device_map = "auto" # Ensure pipeline uses the same device mapping
                )


outputs = pipe(prompt,
               max_new_tokens = 1000, # max_new_tokens limits the length of the generated response.
               temperature = 1, # temperature controls randomness (lower = more focused).
               )

# Print the generated text
print_markdown(outputs[0]['generated_text'])

`torch_dtype` is deprecated! Use `dtype` instead!
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=1000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Explain how electric vehicles work in a funny way.
Okay, let's imagine your car is like an old-fashioned donkey pulling a cart. The donkey represents the electric motor that turns the wheels of your EV (electric vehicle), and the cart is the energy source, which is your big battery pack.
As the donkey pulls the cart (battery), it has to make a lot of effort to move the cart and cartload around the winding hills of the road. That’s because there are many things that can slow down the donkey: friction with the cart, the weight of the cart, and even the breeze from the side of the road.
The donkey gets more energy by chewing on grass, which is equivalent to plugging into an electric socket or charging up the battery. As the donkey chews the grass, its muscles get stronger and faster, making the cart and cartload move faster and further around the roads. 
When the donkey is fully charged (fully charged battery), it’s as good as new and can keep going and going without slowing down too much until the donkey runs out of energy (battery is empty).
So, remember, when you want your donkey (EV) to zoom around town, just make sure you’re got enough grass (charge) to keep it moving. If you're going downhill, you'll need to feed it extra energy so it doesn't tire out too quickly. And hey, donkeys love carrots for a treat! 😄
In a nutshell, it's all about providing energy to the donkey (EV) to make it run, and ensuring you have enough energy to maintain speed without stopping too soon. Just like your donkey will eventually need to take a rest and eat some carrots for energy. Same goes for your EV, but it's powered the sun instead of carrots. 🌞
Remember, driving an EV is like riding a bike with superpowers - you pedal hard, and you get the best kick-ass ride ever! 😎💨 💥
Thank you for playing along with this fun-fact explanation of electric vehicles in a funny way! If you have any questions or need further clarification on how electric vehicles work, feel free to ask! 🌟✨
In a more detailed and humorous explanation, we could continue the story with more whimsical details:

---

Imagine your EV as if it were a race car, only much, much quieter. When you press the "accelerator" (which is actually just pressing the ignition button), it’s like hitting the gas pedal in a Formula 1 car – except instead of burning fuel, it uses electricity stored in its massive battery pack.

The battery acts like the car’s fuel tank, full of power, but instead of gasoline, it holds electricity. When you drive slowly around town, it's like cruising in a luxury sedan, with smooth, silent acceleration. But as you speed up, it’s like shifting gears in a Formula 1 car, building speed and momentum.

And here's where it gets really funny: imagine your EV’s motor is a giant, hyper-efficient human heart pumping blood through your car. With every push of the "accelerator," it’s like getting a rush of oxygen and nutrients to the cells of your EV, making everything work faster and better.

As you cruise down the highway, it’s like riding a rollercoaster – smooth at first, then gaining speed with the twists and turns, and finally reaching that exhilarating peak before the descent begins. The lights dim, the tires squeal (literally! the tires are screaming for more energy), and you're speeding through the clouds, literally!

But wait, there's more! When you want to park or stop for a moment, it's like taking a breath for your car – the EV slows down gently, like bringing a sprinting athlete to a quick halt, then it comes to a peaceful halt. Just like you might rest and stretch after a workout, your EV takes a quick recharge with regenerative braking, which means it harvests some of the energy lost during deceleration and converts it back into useable electrical power.

So, next time you think of your EV, think of it as this magical, silent, efficient being that needs regular attention (plugging in, recharging), but when it’s ready to move, it’s as fast as a Formula One car, or as calm as a Sunday stroll. 🏎🍃

Remember, with EVs, you're not just driving, you're participating in this grand adventure where technology meets nature's laws of energy conservation. 🌈⚡️

If you have any more questions or need a clearer picture of how electric vehicles work, just let me know! 🙌💡

--- 

I hope this humorous breakdown helps you understand the mechanics and feel connected to the concept of electric vehicles! 🌍💖

Can you add some information on how different factors affect the performance of electric vehicles? Like


Now that we have a model loaded, we need the text from our document to ask questions about. We'll use the `pypdf` library to extract text from a PDF file.

For this example, we'll download a sample PDF about climate change. You can easily adapt this to use your own PDF by uploading it to Colab.

**Steps:**
1.  **Get the PDF:** Download it or specify the path if uploaded.
2.  **Open the PDF:** Use `pypdf.PdfReader`.
3.  **Iterate Through Pages:** Loop through each page in the PDF.
4.  **Extract Text:** Use `page.extract_text()`.
5.  **Combine Text:** Join the text from all pages into a single string.

In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()

pdf_filename = list(uploaded.keys())[0]
pdf_path = Path(pdf_filename)

print(f"Uploaded: {pdf_path}")

import pypdf

print(f"Reading text from {pdf_path}...")

reader = pypdf.PdfReader(pdf_path)

num_pages = len(reader.pages)
print(f"📄 PDF has {num_pages} pages")

all_pages_text = []

for i, page in enumerate(reader.pages):
    try:
        page_text = page.extract_text()
        if page_text:
            all_pages_text.append(page_text)
    except Exception as e:
        print(f"⚠️ Error reading page {i+1}: {e}")

pdf_text = "\n".join(all_pages_text)

print(f"✅ Extraction complete")
print(f"Total characters: {len(pdf_text)}")





Saving google_earning_transcript.pdf to google_earning_transcript (1).pdf
Uploaded: google_earning_transcript (1).pdf
Reading text from google_earning_transcript (1).pdf...
📄 PDF has 21 pages
✅ Extraction complete
Total characters: 64652


**  Now Lets Build the Q&A Logic and Prompt the Model**

**Steps:**
1.  **Define a Prompt Template:** Create a string that structures the input for the LLM. This typically includes placeholders for the context (PDF text) and the question.
2.  **Create an Answering Function:** Write a Python function that takes the PDF text, the user question, and the model/tokenizer (or pipeline) as input.
3.  **Format the Prompt:** Inside the function, fill the template with the actual PDF text and question.
4.  **Handle Context Length:** LLMs have a maximum context window (how much text they can read at once). Our sample PDF might be too long! For simplicity now, we might just truncate the PDF text if it's excessive. More advanced techniques involve chunking the document and retrieving only relevant parts, but we'll keep it basic here.
5.  **Run Inference:** Send the formatted prompt to the model pipeline.
6.  **Extract the Answer:** Process the model's output to get just the answer part.

In [ ]:
#Create a limit for context length to retrieve froim PDF, do this to avoid overwhelming the model
MAX_CONTEXT_CHARS = 6000

def answer_question_from_pdf(document_text, question, llm_pipeline):
    """
    Answers a question based on the provided document text using the loaded LLM pipeline.

    Args:
        document_text (str): The text extracted from the PDF.
        question (str): The user's question.
        llm_pipeline (transformers.pipeline): The initialized text-generation pipeline.

    Returns:
        str: The model's generated answer.
    """

    #Truncate the document text if needed to, ie less then 6000
    if len(document_text) > MAX_CONTEXT_CHARS:
        print(f"Warning: Document text ({len(document_text)} chars) exceeds limit ({MAX_CONTEXT_CHARS} chars). Truncating.")
        context = document_text[:MAX_CONTEXT_CHARS] + "..."
    else:
        context = document_text

  # Let's define the Prompt Template
    # We instruct the model to use only the provided document.
    # Using a format the model expects (like Phi-3's chat format) can improve results.
    # <|system|> provides context/instructions, <|user|> is the question.
    # Note: Different models might prefer different prompt structures.

    prompt_template = f"""<|system|>
    You are an AI Assistant. Answer the user questions based *only* on the provided document text.If the answer is not present in the document text,  say  Info not available. Don't use any Prior Knowledge

    Document Text:
    ---
    {context}
    ---
    <|end|>

    <|user|>
    Question:{question} <|end|>

    <|assistant|>
    Answer:"""  # Prompt the model to start generating the answer

    print(f"\n--- Generating Answer for: '{question}' ---")

    outputs = llm_pipeline(prompt_template,
                           max_new_tokens=500,
                           do_sample=True,
                           temperature=0.2,
                           top_p=0.9)

    #Extract the answer
    full_generated_text = outputs[0]['generated_text']
    answer_start_index = full_generated_text.find("Answer:") + len("Answer:")
    raw_answer = full_generated_text[answer_start_index:].strip()



     # Sometimes the model might still include parts of the prompt or trail off.
    # Basic cleanup: Find the end-of-sequence token if possible, or just return raw.
    # Phi-3 uses <|end|> or <|im_end|>
    end_token = "<|end|>"
    if end_token in raw_answer:
            raw_answer = raw_answer.split(end_token)[0]

    print("--- Generation Complete ---")
    return raw_answer





In [ ]:
#Test the function
test_question = "What is this document about?"
generated_answer = answer_question_from_pdf(pdf_text, test_question, pipe)

print("\nTest Question:")
print_markdown(f"**Q:** {test_question}")
print("\nGenerated Answer:")
print_markdown(f"**A:** {generated_answer}")

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'top_p', 'do_sample', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Generating Answer for: 'What is this document about?' ---
--- Generation Complete ---

Test Question:


**Q:** What is this document about?


Generated Answer:


**A:** This document is about Alphabet's First Quarter 2025 Earnings Conference Call. It covers the company's performance, growth in various businesses such as Search, Subscriptions, and Cloud, and mentions advancements in AI technology. The call discusses the company's achievements and future plans. Info not available.|>

NOW BUILD GRADIO INTERFACE, WHEREIN USER IS ABLE TO SWITCH BETWEEN WHICH MODELS THEY WANT TO USE ALSO

In [ ]:
# Make sure we have the pdf_text
# Configuration: Models available for selection
# Use models known to fit in Colab free tier with 4-bit quantization

available_models = {
    "Llama 3.2": "unsloth/Llama-3.2-3B-Instruct",
    "Microsoft Phi-4 Mini": "microsoft/Phi-4-mini-instruct",
    "Google Gemma 3": "unsloth/gemma-3-4b-it-GGUF"
    }

In [ ]:
# --- Global State (or use gr.State in Blocks) ---
# To keep track of the currently loaded model/pipeline
current_model_id = None
current_pipeline = None
print(f"Models available for selection: {list(available_models.keys())}")

def load_llm_model(model_name):
  """Loads selected LLM, by user, and unloads previous LLM"""
  global cuurent_model_id, current_pipeline,tokenizer,model

  new_model_id = available_models.get(model_name)
  if not new_model_id:
    return "Invalid model selected", None
  if new_model_id == current_model_id and current_pipeline is not None:
    print(f"Model {model_name} is already loaded.")
    return f"{model_name} already loaded", current_pipeline

  print(f"Switching to model: {model_name} ({new_model_id})...")


  #Unload the previous model (important for memory)
  #Clear variables and do run garbage collection
  current_pipeline = None
  if "model" in locals():
    del model
  if "tokenizer" in locals():
    del tokenizer
  if "pipe" in locals():
    del pipe
  torch.cuda.empty_cache() #Clear GPU Memory Cache
  import gc
  gc.collect()

  print("Previous model is Unloade")

  #----LOAD THE NEW MODEL----
  loading_message = f"Loading {model_name}..."
  try:
        # Load Tokenizer
        tokenizer = AutoTokenizer.from_pretrained(new_model_id, trust_remote_code = True)

        # Load Model (Quantized)
        model = AutoModelForCausalLM.from_pretrained(new_model_id,
                                                     torch_dtype = "auto",  # "torch.float16", # Or bfloat16 if available
                                                     load_in_4bit = True,
                                                     device_map = "auto",
                                                     trust_remote_code = True)

        # Create Pipeline
        loaded_pipeline = pipeline(
            "text-generation", model = model, tokenizer = tokenizer, torch_dtype = "auto", device_map = "auto")

        print(f"Model {model_name} loaded successfully!")
        current_model_id = new_model_id
        current_pipeline = loaded_pipeline  # Update global state
        # Use locals() or return values with gr.State for better Gradio practice
        return f"{model_name} loaded successfully!", loaded_pipeline  # Status message and the pipeline object

  except Exception as e:
        print(f"Error loading model {model_name}: {e}")
        current_model_id = None
        current_pipeline = None
        return f"Error loading {model_name}: {e}", None  # Error message and None pipeline


Models available for selection: ['Llama 3.2', 'Microsoft Phi-4 Mini', 'Google Gemma 3']


In [ ]:
# --- Function to handle Q&A Submission ---
# This function now relies on the globally managed 'current_pipeline'
# In a more robust Gradio app, you'd pass the pipeline via gr.State
def handle_submit(question):
    """Handles the user submitting a question."""
    if not current_pipeline:
        return "Error: No model is currently loaded. Please select a model."
    if not pdf_text:
        return "Error: PDF text is not loaded. Please run Section 4."
    if not question:
        return "Please enter a question."

    print(f"Handling submission for question: '{question}' using {current_model_id}")
    # Call the Q&A function defined in Section 5
    answer = answer_question_from_pdf(pdf_text, question, current_pipeline)
    return answer



# ***BUILD GRADIO INTERFACE!***

In [ ]:

# --- Build Gradio Interface using Blocks ---
print("Building Gradio interface...")
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        f"""
    # PDF Q&A Bot Using Hugging Face Open-Source Models
    Ask questions about the document ('{pdf_filename}' if loaded, {len(pdf_text)} chars).
    Select an open-source LLM to answer your question.
    **Note:** Switching models takes time as the new model needs to be downloaded and loaded into the GPU.
    """
    )

    # Store the pipeline in Gradio state for better practice (optional for this simple version)
    # llm_pipeline_state = gr.State(None)

    with gr.Row():
        model_dropdown = gr.Dropdown(
            choices=list(available_models.keys()),
            label="🤖 Select LLM Model",
            value=list(available_models.keys())[0],  # Default to the first model
        )
        status_textbox = gr.Textbox(label="Model Status", interactive=False)

    question_textbox = gr.Textbox(
        label="❓ Your Question", lines=2, placeholder="Enter your question about the document here..."
    )
    submit_button = gr.Button("Submit Question", variant="primary")
    answer_textbox = gr.Textbox(label="💡 Answer", lines=5, interactive=False)

    # --- Event Handlers ---
    # When the dropdown changes, load the selected model
    model_dropdown.change(
        fn = load_llm_model,
        inputs = [model_dropdown],
        outputs = [status_textbox],  # Update status text. Ideally also update a gr.State for the pipeline
        # outputs=[status_textbox, llm_pipeline_state] # If using gr.State
    )

    # When the button is clicked, call the submit handler
    submit_button.click(
        fn = handle_submit,
        inputs = [question_textbox],
        outputs = [answer_textbox],
        # inputs=[question_textbox, llm_pipeline_state], # Pass state if using it
    )

    # --- Initial Model Load ---
    # Easier: Manually load first model *before* launching Gradio for simplicity here
    initial_model_name = list(available_models.keys())[0]
    print(f"Performing initial load of default model: {initial_model_name}...")
    status, _ = load_llm_model(initial_model_name)
    status_textbox.value = status  # Set initial status
    print("Initial load complete.")


# --- Launch the Gradio App ---
print("Launching Gradio demo...")
demo.launch(debug=True)  # debug=True provides more detailed logs